# Chapter 03 — CSV, JSON, and Excel Data Handling

Files are common boundaries between systems. Learn text encoding, CSV delimiters and quoting, JSON objects and arrays, Excel workbooks, and robust import/export patterns.

## Prerequisites
Python file handling and basic DataFrame concepts.

## Learning goals
- Read and write CSV with the standard library and Pandas.
- Parse JSON objects and arrays and understand serialization options.
- Import/export Excel workbooks with Pandas and OpenPyXL.
- Handle encodings, missing values, and data types.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. CSV structure and standard-library API
2. JSON data model
3. Excel workbooks
4. Pandas CSV import parameters
5. Data contracts and export choices

### CSV structure and standard-library API
CSV is a text interchange format, not a database; it has no universal type system. Fields use a delimiter and may be quoted to protect embedded delimiters/newlines. `csv.reader` yields lists; `DictReader` maps fields to header names; `writer` and `DictWriter` export rows. Important options include `delimiter`, `quotechar`, `escapechar`, `fieldnames`, `restkey`, `restval`, `encoding`, and `newline=""` when writing.

#### Demonstration 1 — Write and read CSV dictionaries

Run the code and inspect both the output and the assumptions behind it.

In [2]:
import csv
from pathlib import Path
p=Path("mini.csv")
with p.open("w",newline="",encoding="utf-8") as f:
 w=csv.DictWriter(f,fieldnames=["item","quantity"]); w.writeheader(); w.writerows([{"item":"Pen","quantity":10},{"item":"Book","quantity":4}])
with p.open(encoding="utf-8",newline="") as f: print(list(csv.DictReader(f)))

[{'item': 'Pen', 'quantity': '10'}, {'item': 'Book', 'quantity': '4'}]


### JSON data model
JSON supports objects, arrays, strings, numbers, booleans, and null. `json.load(file)`/`loads(text)` parse; `dump(obj,file)`/`dumps(obj)` serialize. Options: `indent` controls pretty formatting; `sort_keys` stabilizes key order; `ensure_ascii=False` preserves readable Unicode; `separators` controls whitespace; `default=` converts otherwise unsupported objects; `allow_nan=False` rejects non-standard NaN/Infinity. JSON object keys are strings.

#### Demonstration 2 — Serialize JSON

Run the code and inspect both the output and the assumptions behind it.

In [3]:
import json
payload={"city":"Kolkata","temperature":28.5,"raining":False}
text=json.dumps(payload,ensure_ascii=False,indent=2,sort_keys=True)
print(text); restored=json.loads(text); print(type(restored).__name__,restored["city"])

{
  "city": "Kolkata",
  "raining": false,
  "temperature": 28.5
}
dict Kolkata


### Excel workbooks
An `.xlsx` workbook contains sheets, cells, formulas, formatting, and other workbook objects. `pd.read_excel(io, sheet_name=0, header=0, usecols=None, dtype=None, skiprows=None, nrows=None, engine=None)` returns a DataFrame or a dict when `sheet_name=None`. `to_excel` exports a frame; `ExcelWriter` writes multiple sheets. OpenPyXL is useful for cell-level styles, formulas, dimensions, and charts.

#### Demonstration 3 — Read CSV with Pandas

Run the code and inspect both the output and the assumptions behind it.

In [4]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
df=pd.read_csv(DATA/"sales.csv",dtype={"order_id":"string"},parse_dates=["order_date"])
print(df.shape); display(df.head(3)); print(df.dtypes)

Data directory: /mnt/data/python_data_analysis_book/data
(240, 11)
  order_id order_date region  ...       cost    profit  customer_id
0    O0101 2025-01-02  North  ...  139372.86  80562.93         C011
1    O0143 2025-01-03   West  ...    7802.99   4092.40         C008
2    O0120 2025-01-04   West  ...   13006.05   3653.15         C015

[3 rows x 11 columns]
order_id       string[python]
order_date     datetime64[ns]
region                 object
product                object
units                   int64
unit_price            float64
discount              float64
revenue               float64
cost                  float64
profit                float64
customer_id            object
dtype: object


### Pandas CSV import parameters
`read_csv` options commonly used: `sep`, `header`, `names`, `index_col`, `usecols`, `dtype`, `parse_dates`, `na_values`, `keep_default_na`, `encoding`, `encoding_errors`, `decimal`, `thousands`, `skiprows`, `nrows`, and `on_bad_lines`. Choose `dtype` for IDs and `parse_dates` for timestamps. Inspect inferred types after import.

#### Demonstration 4 — Read a JSON array

Run the code and inspect both the output and the assumptions behind it.

In [5]:
import json, pandas as pd
records=json.loads((DATA/"weather.json").read_text(encoding="utf-8"))
weather=pd.DataFrame.from_records(records); display(weather)

         date     city  temperature_c  humidity_pct condition
0  2025-01-01  Kolkata           22.5            68     Clear
1  2025-01-02  Kolkata           23.1            65    Cloudy
2  2025-01-03    Delhi           14.2            42       Fog
3  2025-01-04   Mumbai           28.4            76     Humid


### Data contracts and export choices
Blanks, `NA`, `N/A`, and zero can have different meanings. Validate required columns and expected types. CSV is widely interoperable; JSON handles nested records; Excel is convenient for formatted multi-sheet handoffs. Preserve raw files and document how missing markers, date formats, and decimal separators are handled.

#### Demonstration 5 — Write a multi-sheet workbook

Run the code and inspect both the output and the assumptions behind it.

In [6]:
with pd.ExcelWriter("sample_report.xlsx",engine="openpyxl") as writer:
 df.head(20).to_excel(writer,sheet_name="Orders",index=False)
 weather.to_excel(writer,sheet_name="Weather",index=False)
print(pd.read_excel("sample_report.xlsx",sheet_name="Orders",usecols=["order_id","region","revenue"]).head())

  order_id region    revenue
0    O0101  North  219935.79
1    O0143   West   11895.39
2    O0120   West   16659.20
3    O0188  South   15634.04
4    O0047   East    8934.03


## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: CSV structure and standard-library API.
- Explain one important concept from: JSON data model.
- Explain one important concept from: Excel workbooks.
- Explain one important concept from: Pandas CSV import parameters.
- Explain one important concept from: Data contracts and export choices.